# Chapter 03: Quantum Gates — The Physics of Control

## Introduction

Welcome to **Section 1.3: General Lecture on Quantum Technology**.

We have a qubit.
It is sitting in a refrigerator.
It is cold.
It is isolated.
Currently, it is useless.
A qubit that does nothing is just a fancy rock.

To compute, we must change it.
We must grab the state vector and rotate it.
We must point it where we want.
We call these operations **Quantum Gates**.

In classical computers, a gate is a physical thing.
A NAND gate is a cluster of transistors.
In quantum computers, a gate is an **Action**.
It is a pulse of energy.
It is a microwave tone sent down a wire.
It is a laser blast lasting 10 nanoseconds.

**The Structure of this Notebook:**
1.  **The Bloch Sphere Rotations:** The geometry of logic.
2.  **The Pauli Gates:** X, Y, Z.
3.  **The Superposition Gate:** Hadamard.
4.  **Microwave Physics:** How to talk to an atom.
5.  **Rabi Oscillations:** The heartbeat of a quantum computer.
6.  **Pulse Shaping:** Why square pulses are bad.
7.  **Two-Qubit Gates:** Making them talk (CNOT, CZ).
8.  **Quantum Process Tomography:** How do we know it worked?
9.  **Randomized Benchmarking:** Measuring the quality.
10. **The Mathematics of Unitaries:** Deriving the matrices.
11. **Visualization:** Plotting the Bloch Sphere.
12. **Python Exercise:** Building a Pulse Scheduler.
13. **Quantum Optimal Control:** Pushing the limits.
14. **The Clifford Group:** A special subset.
15. **Error Channels:** How gates fail.
16. **Dynamic Decoupling:** Keeping memory alive.
17. **Cryogenic Control:** Electronics at 4 Kelvin.

Let us begin.

---

## Part 1: The Bloch Sphere Rotations

**The Geometry**
The qubit is a vector on a sphere.
Any operation on a single qubit is a rotation.
You can rotate around the X-axis.
You can rotate around the Y-axis.
You can rotate around the Z-axis.

**The Math (Unitary Matrices)**
Quantum mechanics is linear algebra.
A state is a vector $|\psi\rangle$.
A gate is a matrix $U$.
The operation is matrix multiplication:
$$ |\psi_{new}\rangle = U |\psi_{old}\rangle $$
The matrix must be **Unitary** ($U^\dagger U = I$).
This means it preserves the length of the vector.
It preserves probability.
It is reversible.
You can always undo a quantum gate.
This is different from classical gates (AND, OR) which destroy information.

---

## Part 2: The Pauli Gates (X, Y, Z)

**The X Gate (The Bit Flip)**
This is the quantum NOT gate.
It turns $|0\rangle$ to $|1\rangle$.
It turns $|1\rangle$ to $|0\rangle$.
On the Bloch sphere, it is a $180^\circ$ rotation around the X-axis.
$$ X = \begin{bmatrix} 0 & 1 \\ 1 & 0 \end{bmatrix} $$

**The Z Gate (The Phase Flip)**
This does nothing to $|0\rangle$.
But it adds a minus sign to $|1\rangle$.
$$ Z|0\rangle = |0\rangle $$
$$ Z|1\rangle = -|1\rangle $$
On the sphere, it rotates around the Z-axis (the vertical pole).
It changes the phase $\phi$.
It is crucial for interference.
$$ Z = \begin{bmatrix} 1 & 0 \\ 0 & -1 \end{bmatrix} $$

**The Y Gate**
It combines bit flip and phase flip.
$$ Y = \begin{bmatrix} 0 & -i \\ i & 0 \end{bmatrix} $$

---

## Part 3: The Hadamard Gate (Superposition)

**The Master Key**
This is the most important gate.
It has no classical analog.
It takes a definite state and creates superposition.
$$ H|0\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}} = |+\rangle $$
$$ H|1\rangle = \frac{|0\rangle - |1\rangle}{\sqrt{2}} = |-\rangle $$
It puts the qubit on the Equator.
It gives it a 50/50 chance of being 0 or 1.
We use it at the start of almost every algorithm.
$$ H = \frac{1}{\sqrt{2}} \begin{bmatrix} 1 & 1 \\ 1 & -1 \end{bmatrix} $$

---

## Part 4: Microwave Physics (How to actually do it)

**The Resonant Drive**
Okay, the math says "Multiply by Matrix X".
But how do we tell the atom to do that?
We use a microwave pulse.
The qubit has a frequency $\omega_q$ (e.g., 5 GHz).
We send a signal at exactly 5 GHz.
$$ V(t) = A(t) \cos(\omega_d t + \phi) $$

**The Interaction Frame**
If the drive frequency $\omega_d$ matches the qubit frequency $\omega_q$, magic happens.
In the rotating frame, the fast oscillations disappear.
The microwave field looks like a static magnetic field.
This field points in the XY plane.
The qubit precesses around this field.
This precession is the rotation.

**The Amplitude determines the Speed**
If you shout louder (higher Amplitude $A$), the qubit rotates faster.
This rotation speed is called the **Rabi Frequency** ($\Omega$).
The angle of rotation is $\theta = \Omega \times t$.
If you want a 180 degree rotation (X gate), you pick a time $t = \pi / \Omega$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# PART 5 CODE: RABI OSCILLATIONS
# Simulating the population transfer under a resonant drive.

# Constants
h_bar = 1.0
Omega = 1.0 # Rabi Frequency (arbitrary units)

# Time evolution
t = np.linspace(0, 4*np.pi, 200)

# Probability of being in |1> (Excited State)
# P1 = sin^2(Omega * t / 2)
P1 = np.sin(Omega * t / 2)**2
P0 = 1 - P1

plt.figure(figsize=(10, 6))
plt.plot(t, P1, label='Excited State |1>', linewidth=3, color='red')
plt.plot(t, P0, '--', label='Ground State |0>', linewidth=2, color='blue')

# Mark Gates
pi_time = np.pi / Omega
plt.axvline(pi_time, color='black', linestyle=':', label='Pi Pulse (X Gate)')
plt.axvline(pi_time/2, color='green', linestyle=':', label='Pi/2 Pulse (H Gate equivalent)')

# Styling
plt.xlabel("Time (t)")
plt.ylabel("Probability")
plt.title("Rabi Oscillations: Driving the Qubit")
plt.legend()
plt.grid(True)
plt.ylim(-0.1, 1.1)

plt.show()

---

## Part 6: Pulse Shaping (DRAG)

**The Spectral Leakage Problem**
If you use a square pulse (turn on, wait, turn off), you have sharp edges.
Sharp edges in time = Broad spectrum in frequency (Fourier Transform).
Wait, our qubit is an Anharmonic Oscillator.
The $|1\rangle \to |2\rangle$ transition is only 300 MHz away.
If your pulse is too sharp, you might hit the $|2\rangle$ state by accident.
This is "Leakage".
It kills the calculation.

**The Gaussian Pulse**
To fix this, we smooth the edges.
We use a Gaussian (Bell curve) shape.
This minimizes the frequency spread.
But it's not enough.

**DRAG (Derivative Removal by Adiabatic Gate)**
This is a clever trick found by IBM/Yale.
We add a second drive perpendicular to the first one.
This second drive is proportional to the derivative of the Gaussian.
It cancels out the leakage to the $|2\rangle$ state.
It allows us to drive the qubit extremely fast (20ns) without errors.

---

## Part 7: Two-Qubit Gates (Entanglement)

**The CNOT Gate**
Controlled-NOT.
If Control is 0, Target stays matching.
If Control is 1, Target flips.
This creates entanglement.
$$ CNOT |00\rangle = |00\rangle $$
$$ CNOT |10\rangle = |11\rangle $$

**Physics of CNOT (Cross-Resonance)**
IBM uses the Cross-Resonance Effect.
We drive Qubit 1 at the frequency of Qubit 2.
Because they are weakly coupled, this drives a rotation on Qubit 2.
The direction of rotation depends on the state of Qubit 1.
It works entirely with microwaves.
No tunable couplers needed.

**The CZ Gate (Controlled-Phase)**
Google uses the CZ gate.
It adds a minus sign only to the $|11\rangle$ state.
To do this, they tune the frequency of Qubit 1 to match Qubit 2.
They bring them into resonance.
They interact for a specific time.
Then they pull them apart.
This requires "Flux Tunable" qubits.

---

## Part 8: Quantum Process Tomography

**How do we know the gate worked?**
You apply an X gate.
You measure 1.
Was it perfect?
Or did it overshoot to 1.1?
Or did it rotate slightly in the Y direction?
Measuring the output state once tells you nothing about the *process*.

**The Method**
To characterize a gate (Operation $U$), we must:
1.  Prepare the qubit in different states ($|0\rangle, |1\rangle, |+\rangle, |i\rangle$).
2.  Apply the gate $U$.
3.  Measure the output in different bases (X, Y, Z).
By correlating all inputs to all outputs, we can reconstruct the "Process Matrix" $\chi$.
This is called Quantum Process Tomography (QPT).
It is slow.
It scales exponentially ($16^N$ measurements).
For 1 qubit, it's fine.
For 50 qubits, it's impossible.

---

## Part 9: Randomized Benchmarking (The Gold Standard)

**The Problem with SPAM**
QPT has a flaw.
It cannot distinguish gate errors from State Preparation And Measurement (SPAM) errors.
If your readout is bad, your gate looks bad.

**The Solution**
Randomized Benchmarking (RB).
Instead of 1 gate, we apply a sequence of random Clifford gates.
Sequence: $C_1, C_2, C_3, ..., C_m$.
At the end, we calculate the inverse of the whole sequence.
We apply the inverse.
The qubit should return to $|0\rangle$.
We measure the probability of $|0\rangle$.
We repeat this for longer and longer sequences ($m=10, 20, 100...$).
The probability will decay exponentially.
The *rate* of decay tells us the average error per gate.
This method is immune to SPAM errors.
This is how IBM reports "99.9% Fidelity".

---

## Part 10: The Mathematics of Unitaries (Deep Dive)

**Schrödinger's Equation**
$$ i\hbar \frac{d}{dt} U(t) = H U(t) $$
If the Hamiltonian $H$ is constant, the solution is:
$$ U(t) = e^{-iHt/\hbar} $$

**Deriving the X-Rotation**
Let $H = \frac{\hbar \Omega}{2} \sigma_x$ (Driving with an X-field).
Then:
$$ R_x(\theta) = e^{-i \frac{\theta}{2} \sigma_x} $$
Using Euler's identity for matrices:
$$ e^{-i A \theta} = I \cos(\theta) - i A \sin(\theta) $$
(where $A^2 = I$).
So:
$$ R_x(\theta) = \begin{bmatrix} \cos(\frac{\theta}{2}) & -i\sin(\frac{\theta}{2}) \\ -i\sin(\frac{\theta}{2}) & \cos(\frac{\theta}{2}) \end{bmatrix} $$
This explains why a $2\pi$ rotation gives $-1$ (global phase).
You need a $4\pi$ rotation to return exactly to $I$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# PART 11 CODE: 3D BLOCH SPHERE VISUALIZATION using Matplotlib

def plot_bloch_vector(theta, phi, title="Qubit State"):
    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(111, projection='3d')

    # Sphere surface
    u, v = np.mgrid[0:2*np.pi:30j, 0:np.pi:15j]
    x = np.cos(u)*np.sin(v)
    y = np.sin(u)*np.sin(v)
    z = np.cos(v)
    ax.plot_wireframe(x, y, z, color="lightgray", alpha=0.3)

    # Axes
    ax.plot([0, 0], [0, 0], [-1, 1], color="black", linewidth=1)
    ax.plot([0, 0], [-1, 1], [0, 0], color="black", linewidth=1)
    ax.plot([-1, 1], [0, 0], [0, 0], color="black", linewidth=1)
    ax.text(0, 0, 1.1, "|0> (Z)", fontsize=12)
    ax.text(0, 0, -1.1, "|1> (-Z)", fontsize=12)
    ax.text(1.1, 0, 0, "|+> (X)", fontsize=12)
    ax.text(0, 1.1, 0, "|i> (Y)", fontsize=12)

    # Vector
    # Convert spherical to cartesian
    vx = np.sin(theta) * np.cos(phi)
    vy = np.sin(theta) * np.sin(phi)
    vz = np.cos(theta)

    ax.quiver(0, 0, 0, vx, vy, vz, color="red", linewidth=3, arrow_length_ratio=0.2)

    ax.set_title(title)
    ax.set_axis_off()
    plt.show()

# Visualize the |+> state
# Theta = pi/2 (Equator), Phi = 0 (X-axis)
plot_bloch_vector(np.pi/2, 0, "The |+> State (Superposition)")


---

## Part 12: Python Exercise: Building a Pulse Scheduler

Let's write a simple Python class to generate Gaussian pulses.
This is a "Hello World" for Pulse Engineering.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class PulseGenerator:
    def __init__(self, sample_rate=1e9): # 1 GigaSample/sec
        self.sample_rate = sample_rate

    def gaussian(self, duration, amp, sigma):
        num_points = int(duration * self.sample_rate)
        t = np.linspace(-duration/2, duration/2, num_points)
        pulse = amp * np.exp(-(t**2) / (2 * sigma**2))
        return t, pulse

    def square(self, duration, amp):
        num_points = int(duration * self.sample_rate)
        return np.linspace(0, duration, num_points), np.full(num_points, amp)

# Settings
duration_ns = 20e-9 # 20 nanoseconds
gen = PulseGenerator()

# Create Pulses
t_gauss, y_gauss = gen.gaussian(duration_ns, amp=1.0, sigma=3e-9)
t_square, y_square = gen.square(duration_ns, amp=1.0)

# Plot Comparison
plt.figure(figsize=(10, 5))
plt.plot(t_gauss*1e9, y_gauss, label="Gaussian Pulse (Low Leakage)", color='blue')
plt.plot(t_square*1e9, y_square, label="Square Pulse (High Leakage)", color='red', linestyle='--')
plt.title("Pulse Shapes for Quantum Control")
plt.xlabel("Time (ns)")
plt.ylabel("Amplitude")
plt.legend()
plt.grid(True)
plt.show()

---

## Part 13: Quantum Optimal Control

**The Problem**
Simple Gaussian pulses work well, but they are not perfect.
Maybe there is noise.
Maybe the Hamiltonian has extra terms.
How do we find the "Perfect Pulse"?

**GRAPE (Gradient Ascent Pulse Engineering)**
This is an optimization algorithm.
1.  Discretize the pulse into 100 pixels.
2.  Calculate the final fidelity.
3.  Calculate the gradient (how changing each pixel improves fidelity).
4.  Update the pixels.
5.  Repeat.
GRAPE finds weird-looking pulses that work perfectly.
They look like squiggly lines.
But they cancel out all the errors.

**CRAB (Chopped Random Basis)**
Similar to GRAPE, but parameterized by frequencies.
It is faster but less flexible.
In 2024, almost all high-fidelity gates are optimized using these algorithms.

---

## Part 14: The Clifford Group

**Definition**
The Clifford Group is the set of gates that normalize the Pauli Group.
This sounds fancy, but it means:
If you apply a Clifford gate to a Pauli matrix (X, Y, Z), you get another Pauli matrix.
Example:
$$ H Z H = X $$
$$ H X H = Z $$

**The Gottesman-Knill Theorem**
Any quantum circuit made ONLY of Clifford gates can be simulated efficiently on a classical computer.
You don't need a quantum computer for:
-   CNOTs
-   Hadamards
-   Phase Gates (S)
-   Measurements
This is shocking.
Entanglement alone is NOT enough for speedup.
You need **Non-Clifford Gates** (like the T-gate) to get exponential speedup.
The T-gate is the "magic" ingredient.
It is also the hardest gate to perform fault-tolerantly.

---

## Part 15: Error Channels

**How do gates fail?**
The environment is constantly trying to destroy your qubit.
We model this using Master Equations.

**The Depolarizing Channel**
This is the "generic" error.
With probability $p$, the qubit is replaced by the maximally mixed state ($I/2$).
This shrinks the Bloch vector towards the center.
It loses all information.

**The Amplitude Damping Channel ($T_1$)**
The qubit loses energy.
It relentlessly falls from $|1\rangle$ to $|0\rangle$.
This is a non-unital channel (it prefers one direction).
On the Bloch sphere, the North Pole ($|0\rangle$) attracts all states.

**The Phase Damping Channel ($T_2$)**
The qubit keeps its energy, but loses its phase.
The vector shrinks towards the Z-axis.
This is caused by magnetic field fluctuations.
It makes quantum interference impossible.

---

## Part 16: Dynamic Decoupling

**The Echo Effect**
Imagine runners on a track.
Some run fast, some run slow.
They spread out (Dephasing).
At time $t$, we shout "Turn Around!" (Apply an X gate).
Now the fast runners are at the back, running towards the start.
The slow runners are at the front, running towards the start.
At time $2t$, they all arrive at the start line together.
Example: The **Hahn Echo**.

**CPMG Sequences**
Carr-Purcell-Meiboom-Gill.
Instead of one X gate, apply a sequence: X - X - X - X.
This constantly refocuses the noise.
It acts as a filter function.
It filters out low-frequency noise.
This is standard practice on all modern QPUs.

---

## Part 17: Cryogenic Control

**The Wiring Bottleneck**
Each qubit needs 2-4 coaxial cables.
A 1000-qubit processor needs 4000 cables.
This is impossible.
Heat travels down the cables.
It will melt the fridge.

**Cryo-CMOS**
We need to put the control electronics INSIDE the fridge.
We need classical chips that work at 4 Kelvin.
Intel (Horse Ridge) and Google are working on this.
The FPGA generates the pulses right next to the qubit.
This reduces the cable count to just a few fibers.
This is the only way to scale to 1,000,000 qubits.

---

## Conclusion

We have gone deep.
We started with simple rotations.
We ended with Optimal Control and Group Theory.
The takeaway is this:
A quantum gate is not magic.
It is a carefully engineered physical process.
It involves detailed calibration, pulse shaping, and error characterization.

In the next notebook, we will zoom out.
We will look at how we arrange thousands of these qubits.
We will look at **Architecture**.

**Navigation:** [Next → Chapter 04: The Architecture](04_computer_architecture.ipynb)

---

## Appendix: Glossary

-   **Clifford Group:** A special set of gates (H, S, CNOT) that can be simulated efficiently classically (Gottesman-Knill theorem).
-   **CPMG:** A sequence of pulses used to cancel out noise (Dynamic Decoupling).
-   **DRAG:** Derivative Removal by Adiabatic Gate. A pulse shaping technique to reduce leakage.
-   **Fidelity:** A measure of how close two quantum states are ($F = |\langle\psi|\phi\rangle|^2$).
-   **Rabi Frequency:** The speed at which a qubit rotates when driven.
-   **Ramsey Fringe:** An interference pattern used to measure the qubit frequency and T2 time.
-   **SPAM:** State Preparation And Measurement errors.
-   **Unitary Matrix:** A complex square matrix where the conjugate transpose is the inverse ($U^\dagger U = I$).